# Lab RAG para Designing Machine Learning Systems

Este notebook inicia a linha de base (baseline RAG). self RAG  e a variante adaptive RAG para comparar:
- qualidade das respostas
- latência
- custo de chamadas ao LLM
- variação entre execuções

A estrutura foi desenhada para usar o PDF local do livro e as 20 perguntas em `questions.json`.


In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
CONFIG_PATH = ROOT / 'configs' / 'experiment.json'
QUESTIONS_PATH = ROOT / 'questions.json'

def load_json(path):
    with open(path, 'r', encoding='utf-8') as fh:
        return json.load(fh)

config = load_json(CONFIG_PATH)
questions = load_json(QUESTIONS_PATH)
print(f"Projeto: {config.get('project_name', 'rag-dmls-lab')}")
print(f"Perguntas carregadas: {len(questions.get('questions', []))}")
print(f"Tema: {config.get('topic', 'Designing Machine Learning Systems')}")


Projeto: rag-dmls-lab
Perguntas carregadas: 20
Tema: Designing Machine Learning Systems


In [2]:
def find_pdf_path(root):
    candidates = [root / 'corpus' / 'documentos', root / 'corpus']
    for folder in candidates:
        if folder.exists():
            matches = sorted(folder.rglob('*.pdf'))
            if matches:
                return matches[0]
    raise FileNotFoundError(
        'PDF do livro não encontrado. Coloque a cópia em corpus/documentos/ ou corpus/.'
    )

def read_pdf_pages(pdf_path):
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        text = page.extract_text() or ''
        pages.append(text)
    return pages

def chunk_text(text, chunk_size=800, overlap=120):
    text = text.replace('\r', '\n')
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(start + chunk_size - overlap, end - overlap)
    return chunks

pdf_path = find_pdf_path(ROOT)
print(f'Arquivo PDF localizado: {pdf_path}')


Arquivo PDF localizado: /mnt/ssd/projeto_rag/rag-dmls-lab/corpus/documentos/designing-machine-learning-systems-2022.pdf


In [3]:
def build_index(chunks):
    vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
    matrix = vectorizer.fit_transform(chunks)
    return vectorizer, matrix

def expand_query(query):
    expansions = {
        'business value': 'business objective success metric problem framing use case',
        'clear definition of the problem': 'problem framing task definition objective success criteria',
        'data-centric approach': 'data-centric AI data quality labels dataset improvement',
        'model-centric approach': 'model-centric AI algorithm architecture model improvement',
        'real-world performance': 'production performance deployment distribution shift noisy inputs',
        'feedback loops': 'degenerate feedback loops popularity bias production predictions',
    }
    lowered = query.lower()
    return ' '.join([query] + [value for key, value in expansions.items() if key in lowered])

def retrieve_top_k(query, chunks, vectorizer, matrix, k=4):
    query = expand_query(query)
    q_vector = vectorizer.transform([query])
    scores = cosine_similarity(q_vector, matrix).ravel()
    ranked = np.argsort(scores)[::-1]
    ranked = [idx for idx in ranked if scores[idx] > 0][:k]
    return [(chunks[idx], float(scores[idx])) for idx in ranked]

GENERATOR_PROMPT_PATH = ROOT / 'prompts' / 'generator.txt'

def load_generator_template(path=None):
    path = Path(path) if path else GENERATOR_PROMPT_PATH
    return path.read_text(encoding='utf-8')

def format_prompt(question, context, template=None):
    template = template or load_generator_template()
    return template.format(question=question, context=context)

def route_question(question):
    q = question.lower()
    adaptive_markers = [
        'compare', 'tradeoff', 'trade-off', 'architecture', 'design', 'monitor',
        'monitoring', 'latency', 'cost', 'observability', 'system', 'serving'
    ]
    return 'adaptive' if any(token in q for token in adaptive_markers) else 'baseline'

print('Funções de busca e roteamento prontas.')


Funções de busca e roteamento prontas.


In [4]:
OLLAMA_URL = 'http://localhost:11434/api/generate'


def call_ollama(prompt, model=None, temperature=None, timeout=120):
    model = model or config.get('generator_model', 'qwen2.5:7b')
    temperature = temperature if temperature is not None else config.get('generation_temperature', 0.0)
    payload = {
        'model': model,
        'prompt': prompt,
        'stream': False,
        'options': {'temperature': temperature},
    }
    response = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
    response.raise_for_status()
    data = response.json()
    return data.get('response', '').strip()


def context_from_hits(hits):
    return '\n\n---\n\n'.join(
        [f'[Passage {i + 1}] {text[:1600]}' for i, (text, _) in enumerate(hits)]
    )


def answer_question_baseline(question, chunks, vectorizer, matrix, k=None, model=None):
    k = k or config.get('top_k', 4)
    hits = retrieve_top_k(question, chunks, vectorizer, matrix, k=k)
    if not hits:
        return 'Não encontrei evidência suficiente no corpus para responder com confiança.'
    prompt = format_prompt(question, context_from_hits(hits))
    try:
        return call_ollama(prompt, model=model)
    except Exception as exc:
        return f'Erro ao chamar o Ollama: {exc}'


def answer_question_adaptive(question, chunks, vectorizer, matrix, k=None, model=None):
    k = k or config.get('top_k', 4)
    q = question.lower()
    variants = [expand_query(question)]
    if any(token in q for token in ['compare', 'tradeoff', 'trade-off', 'architecture', 'design', 'monitor', 'monitoring', 'latency', 'cost', 'observability', 'system', 'serving']):
        variants.extend([
            'trade-offs in designing machine learning systems',
            'monitoring, evaluation and operational constraints in ML',
            'data quality and deployment considerations in ML systems'
        ])
    context_parts = []
    seen = set()
    for variant in variants:
        for text, score in retrieve_top_k(variant, chunks, vectorizer, matrix, k=k):
            signature = text[:200]
            if signature not in seen:
                seen.add(signature)
                context_parts.append((text, score))
    if not context_parts:
        return 'Não encontrei evidência suficiente no corpus para responder com confiança.'
    prompt = format_prompt(question, context_from_hits(context_parts[:k]))
    try:
        return call_ollama(prompt, model=model)
    except Exception as exc:
        return f'Erro ao chamar o Ollama: {exc}'


def parse_self_rag_critique(raw_critique):
    try:
        parsed = json.loads(raw_critique)
    except json.JSONDecodeError:
        start = raw_critique.find('{')
        end = raw_critique.rfind('}')
        if start < 0 or end <= start:
            raise ValueError('A crítica do Self-RAG não retornou JSON válido.') from None
        try:
            parsed = json.loads(raw_critique[start:end + 1])
        except json.JSONDecodeError as exc:
            raise ValueError('A crítica do Self-RAG não retornou JSON válido.') from exc
    required = {'relevant', 'supported', 'needs_more_evidence'}
    if not required.issubset(parsed):
        raise ValueError('A crítica do Self-RAG não contém os campos obrigatórios.')
    return {
        'relevant': bool(parsed['relevant']),
        'supported': bool(parsed['supported']),
        'needs_more_evidence': bool(parsed['needs_more_evidence']),
        'reason': str(parsed.get('reason', '')),
    }


def self_rag_critique(question, draft, context, model=None):
    prompt = (
        'You are a strict verifier for a Self-RAG pipeline. Return only a JSON object '
        'with boolean fields relevant, supported, needs_more_evidence and a short reason. '
        'relevant means the passages help answer the question. supported means the draft '
        'is supported by the passages and does not invent facts. needs_more_evidence means '
        'another retrieval round is needed.\n\n'
        f'Question: {question}\n\nContext:\n{context}\n\nDraft answer:\n{draft}\n\nJSON:'
    )
    return parse_self_rag_critique(call_ollama(prompt, model=model))


def self_rag_answer(question, chunks, vectorizer, matrix, k=None, model=None):
    k = k or config.get('top_k', 4)
    initial_hits = retrieve_top_k(question, chunks, vectorizer, matrix, k=k)
    if not initial_hits:
        return {
            'answer': 'Não encontrei evidência suficiente no corpus para responder com confiança.',
            'hits': [],
            'llm_calls': 0,
            'retrieval_rounds': 1,
            'reflection': {'relevant': False, 'supported': False, 'needs_more_evidence': True, 'reason': 'No initial evidence.'},
        }

    context = context_from_hits(initial_hits)
    draft = call_ollama(format_prompt(question, context), model=model)
    critique = self_rag_critique(question, draft, context, model=model)
    llm_calls = 2
    hits = initial_hits

    if critique['supported'] and critique['relevant'] and not critique['needs_more_evidence']:
        answer = draft
    else:
        expanded_query = f'{question} evidence limitations counterexamples production context'
        additional_hits = retrieve_top_k(expanded_query, chunks, vectorizer, matrix, k=k)
        seen = {text[:200] for text, _ in hits}
        hits = hits + [(text, score) for text, score in additional_hits if text[:200] not in seen]
        hits = hits[:config.get('max_context_chunks', k)]
        final_context = context_from_hits(hits)
        answer = call_ollama(format_prompt(question, final_context), model=model)
        llm_calls += 1
        critique['retrieval_triggered'] = True

    return {
        'answer': answer,
        'hits': hits,
        'llm_calls': llm_calls,
        'retrieval_rounds': 2 if llm_calls == 3 else 1,
        'reflection': critique,
    }

print('Chamada ao LLM, baseline/adaptive e Self-RAG prontos.')


Chamada ao LLM, baseline/adaptive e Self-RAG prontos.


In [5]:
# SMOKE TEST — validação rápida do pipeline; não gera resultados oficiais.
# Execute esta célula antes do experimento oficial. Ela usa somente as primeiras
# perguntas e mantém os resultados em memória.
def run_demo_experiment():
    raw_pages = read_pdf_pages(pdf_path)
    raw_text = '\n\n'.join(raw_pages)
    chunk_size = config.get('chunk_size_chars', 800)
    overlap = config.get('chunk_overlap_chars', 120)
    chunks = chunk_text(raw_text, chunk_size=chunk_size, overlap=overlap)
    vectorizer, matrix = build_index(chunks)
    dataset = questions.get('questions', [])
    results = []

    for item in dataset[:5]:
        question = item['question']
        start = time.time()
        route = route_question(question)
        answer = (
            answer_question_adaptive(question, chunks, vectorizer, matrix)
            if route == 'adaptive'
            else answer_question_baseline(question, chunks, vectorizer, matrix)
        )
        elapsed = time.time() - start
        results.append({
            'id': item['id'],
            'route': route,
            'latency_sec': round(elapsed, 3),
            'answer': answer
        })

    return pd.DataFrame(results)

sample_df = run_demo_experiment()

# Critérios mínimos para liberar o experimento oficial.
assert len(sample_df) == min(5, len(questions.get('questions', [])))
assert not sample_df.empty, 'O smoke test não produziu nenhuma resposta.'
assert sample_df['answer'].fillna('').str.strip().ne('').all(), (
    'O smoke test produziu pelo menos uma resposta vazia.'
)
assert sample_df['latency_sec'].ge(0).all(), 'Foi registrada latência inválida.'

print(f'SMOKE TEST PASSOU: {len(sample_df)} respostas em memória; nenhum arquivo oficial foi gerado.')
sample_df.head()


SMOKE TEST PASSOU: 5 respostas em memória; nenhum arquivo oficial foi gerado.


,id,route,latency_sec,answer
0,Q01,baseline,12.628,The book emphasizes starting with data quality...
1,Q02,adaptive,19.759,The book describes the debate between improvin...
2,Q03,adaptive,9.047,Building a simple baseline is essential before...
3,Q04,adaptive,17.632,Data quality problems such as label noise and ...
4,Q05,adaptive,17.143,Hidden feedback loops are degenerate feedback ...


## Experimento oficial

Execute esta seção somente depois que o **SMOKE TEST PASSOU**.

A execução oficial percorre as 20 perguntas, respeita as repetições definidas
em `configs/experiment.json` e grava CSVs, traces JSONL e um manifesto em
`results/`. Diferentemente da demonstração, estes arquivos são resultados
persistentes e versionáveis. A função abaixo não é chamada automaticamente.

In [6]:
import hashlib
from datetime import datetime, timezone

RESULTS_DIR = ROOT / 'results'
TRACES_DIR = RESULTS_DIR / 'traces'


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as fh:
        for block in iter(lambda: fh.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def official_hits(question, chunks, vectorizer, matrix, route):
    if route == 'adaptive':
        variants = [expand_query(question)]
        q = question.lower()
        if any(token in q for token in [
            'compare', 'tradeoff', 'trade-off', 'architecture', 'design',
            'monitor', 'monitoring', 'latency', 'cost', 'observability',
            'system', 'serving'
        ]):
            variants.extend([
                'trade-offs in designing machine learning systems',
                'monitoring, evaluation and operational constraints in ML',
                'data quality and deployment considerations in ML systems',
            ])
        candidates = []
        seen = set()
        for variant in variants:
            for text, score in retrieve_top_k(variant, chunks, vectorizer, matrix):
                signature = text[:200]
                if signature not in seen:
                    seen.add(signature)
                    candidates.append((text, score))
        return candidates[:config.get('max_context_chunks', config.get('top_k', 4))]
    return retrieve_top_k(question, chunks, vectorizer, matrix)


def run_official_experiment():
    RESULTS_DIR.mkdir(exist_ok=True)
    TRACES_DIR.mkdir(exist_ok=True)

    raw_text = '\n\n'.join(read_pdf_pages(pdf_path))
    chunks = chunk_text(
        raw_text,
        chunk_size=config.get('chunk_size_chars', 800),
        overlap=config.get('chunk_overlap_chars', 120),
    )
    vectorizer, matrix = build_index(chunks)
    dataset = questions.get('questions', [])
    if len(dataset) != config.get('question_count', len(dataset)):
        raise ValueError(
            f"Quantidade inesperada de perguntas: {len(dataset)} "
            f"(esperado: {config.get('question_count')})."
        )

    all_rows = []
    variants = {
        'baseline': config.get('baseline_repetitions', 3),
        'adaptive': config.get('adaptive_repetitions', 3),
        'self_rag': config.get('self_rag_repetitions', 3),
    }
    trace_path = TRACES_DIR / 'official_runs.jsonl'
    with open(trace_path, 'w', encoding='utf-8') as trace_file:
        for variant, repetitions in variants.items():
            variant_rows = []
            for repetition in range(1, repetitions + 1):
                repetition_rows = []
                for item in dataset:
                    question = item['question']
                    start = time.perf_counter()
                    if variant == 'self_rag':
                        self_rag_result = self_rag_answer(question, chunks, vectorizer, matrix)
                        answer = self_rag_result['answer']
                        hits = self_rag_result['hits']
                        llm_calls = self_rag_result['llm_calls']
                        retrieval_rounds = self_rag_result['retrieval_rounds']
                        reflection = self_rag_result['reflection']
                    else:
                        hits = official_hits(question, chunks, vectorizer, matrix, variant)
                        answer = (
                            answer_question_adaptive(question, chunks, vectorizer, matrix)
                            if variant == 'adaptive'
                            else answer_question_baseline(question, chunks, vectorizer, matrix)
                        )
                        llm_calls = 1
                        retrieval_rounds = 1
                        reflection = None
                    evidence = [
                        {'rank': rank, 'score': round(score, 6), 'text': text}
                        for rank, (text, score) in enumerate(hits, start=1)
                    ]
                    row = {
                        'question_id': item['id'],
                        'category': item['category'],
                        'difficulty': item['difficulty'],
                        'variant': variant,
                        'repetition': repetition,
                        'route': variant,
                        'latency_sec': round(time.perf_counter() - start, 3),
                        'evidence_count': len(evidence),
                        'llm_calls': llm_calls,
                        'retrieval_rounds': retrieval_rounds,
                        'reflection_triggered': bool(reflection and reflection.get('retrieval_triggered', False)),
                        'reflection_supported': None if reflection is None else reflection['supported'],
                        'answer': answer,
                    }
                    repetition_rows.append(row)
                    variant_rows.append(row)
                    all_rows.append(row)
                    trace_file.write(json.dumps({**row, 'question': question, 'evidence': evidence, 'reflection': reflection}, ensure_ascii=False) + '\n')

                if variant in {'baseline', 'self_rag'}:
                    filename = f'{variant}_r{repetition}.csv'
                    pd.DataFrame(repetition_rows).to_csv(RESULTS_DIR / filename, index=False)

            if variant == 'adaptive':
                pd.DataFrame(variant_rows).to_csv(
                    RESULTS_DIR / 'results_adaptive.csv', index=False
                )

    results_df = pd.DataFrame(all_rows)
    results_df.groupby(['variant', 'category', 'repetition'], as_index=False).agg(
        mean_latency_sec=('latency_sec', 'mean'),
        mean_evidence_count=('evidence_count', 'mean'),
        mean_llm_calls=('llm_calls', 'mean'),
        reflection_rate=('reflection_triggered', 'mean'),
        questions=('question_id', 'count'),
    ).to_csv(RESULTS_DIR / 'summary.csv', index=False)

    answer_tables = {
        variant: results_df[results_df['variant'] == variant]
        .groupby('question_id')['answer'].first()
        .rename(f'{variant}_answer')
        for variant in variants
    }
    paired = pd.concat(list(answer_tables.values()), axis=1).reset_index()
    paired['baseline_vs_adaptive_differ'] = paired['baseline_answer'] != paired['adaptive_answer']
    paired['baseline_vs_self_rag_differ'] = paired['baseline_answer'] != paired['self_rag_answer']
    paired['adaptive_vs_self_rag_differ'] = paired['adaptive_answer'] != paired['self_rag_answer']
    paired.to_csv(RESULTS_DIR / 'disagreements.csv', index=False)

    manifest = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'config': config,
        'config_sha256': sha256_file(CONFIG_PATH),
        'questions_sha256': sha256_file(QUESTIONS_PATH),
        'corpus_pdf': str(pdf_path.relative_to(ROOT)),
        'corpus_sha256': sha256_file(pdf_path),
        'question_count': len(dataset),
        'chunk_count': len(chunks),
        'variants': variants,
        'trace_file': str(trace_path.relative_to(ROOT)),
    }
    with open(RESULTS_DIR / 'run_manifest.json', 'w', encoding='utf-8') as fh:
        json.dump(manifest, fh, ensure_ascii=False, indent=2)

    print(f'EXPERIMENTO OFICIAL CONCLUÍDO: {len(results_df)} respostas gravadas em {RESULTS_DIR}.')
    return results_df


In [ ]:
# EXECUÇÃO OFICIAL — rode somente após o smoke test passar.
official_df = run_official_experiment()
official_df.head()

In [ ]:
# PÓS-PROCESSAMENTO — valida a correção sem repetir chamadas ao Ollama.
answer_tables = {}
for variant, filename in {
    'baseline': 'baseline_r1.csv',
    'adaptive': 'results_adaptive.csv',
    'self_rag': 'self_rag_r1.csv',
}.items():
    frame = pd.read_csv(RESULTS_DIR / filename)
    answer_tables[variant] = frame.groupby('question_id')['answer'].first().rename(f'{variant}_answer')

paired = pd.concat(list(answer_tables.values()), axis=1).reset_index()
paired['baseline_vs_adaptive_differ'] = paired['baseline_answer'] != paired['adaptive_answer']
paired['baseline_vs_self_rag_differ'] = paired['baseline_answer'] != paired['self_rag_answer']
paired['adaptive_vs_self_rag_differ'] = paired['adaptive_answer'] != paired['self_rag_answer']
paired.to_csv(RESULTS_DIR / 'disagreements.csv', index=False)

assert len(paired) == config['question_count']
assert {'baseline_answer', 'adaptive_answer', 'self_rag_answer'}.issubset(paired.columns)
print(f"DIVERGÊNCIAS RECONSTRUÍDAS: {len(paired)} perguntas; colunas planas validadas.")
paired.head()
